In [102]:
import string

import numpy as np
import  pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /Users/user/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [103]:
df = pd.read_csv('spam.csv', encoding='utf-16')

In [104]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [105]:
df = df.drop(['Unnamed: 2','Unnamed: 3','Unnamed: 4'],axis=1)

In [106]:
df['v1'].value_counts()

v1
ham     4825
spam     747
Name: count, dtype: int64

In [107]:
df.shape

(5572, 2)

In [108]:
df = df.rename(columns={'v1': 'label', 'v2': 'message'})


In [109]:
df.replace({'label': { 'ham':'NotSpan'}}, inplace=True)

,label,message
0,NotSpan,"Go until jurong point, crazy.. Available only ..."
1,NotSpan,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,NotSpan,U dun say so early hor... U c already then say...
4,NotSpan,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,NotSpan,Will �_ b going to esplanade fr home?
5569,NotSpan,"Pity, * was in mood for that. So...any other s..."
5570,NotSpan,The guy did some bitching but I acted like i'd...


In [110]:
df.isna().sum()

label      0
message    0
dtype: int64

In [111]:
df.duplicated().sum()

np.int64(403)

In [112]:
df.drop_duplicates(inplace=True)

In [113]:
from  sklearn.preprocessing import OneHotEncoder

In [114]:
#encoder = OneHotEncoder()

In [115]:
#df = pd.get_dummies(df,columns=['label'],drop_first=True)

In [116]:
#spamMessages = df[df['label_spam']==1]

In [117]:
word_counts = df['message'].str.lower().str.split().explode().value_counts()

In [118]:
word_counts

message
i               2095
to              2055
you             1832
a               1281
the             1223
                ... 
pity,              1
so...any           1
suggestions?       1
bitching           1
rofl.              1
Name: count, Length: 13494, dtype: int64

In [119]:
import re
from nltk.corpus import stopwords

# Create stopword set once
STOP_WORDS = set(stopwords.words('english'))

# Add custom stopwords
STOP_WORDS.update([
    'po'
])

def preprocess_text(text):
    # Handle missing values
    if not isinstance(text, str):
        return ""

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove emails
    text = re.sub(r'\S+@\S+', '', text)

    # Keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenize and remove stopwords
    words = [
        word for word in text.split()
        if word not in STOP_WORDS
    ]

    return " ".join(words)

In [120]:
df['message'] = df['message'].apply(preprocess_text)

In [121]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [122]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# Features and target
X = df['message']
y = df['label']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Create training pipeline
model = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1,2),
        min_df=2
    )),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9535783365570599
              precision    recall  f1-score   support

     NotSpan       0.95      1.00      0.97       903
        spam       0.99      0.64      0.78       131

    accuracy                           0.95      1034
   macro avg       0.97      0.82      0.88      1034
weighted avg       0.96      0.95      0.95      1034



In [123]:
model.predict([
    "Congratulations! You have won a free iPhone. Call now!"
])

array(['spam'], dtype=object)

In [124]:
from joblib import dump
dump(model, 'SMSspamClassificationModel.pkl')

['SMSspamClassificationModel.pkl']